# Exact lossy polariton model — from scratch

This notebook implements the Lorentzian-loss specialization of Cortese & De Liberato,
*Exact solution of polaritonic systems with arbitrary light and matter frequency-dependent losses*
(JCP 156, 084106, 2022; arXiv:2111.05258).

**Dependencies:** NumPy and Matplotlib only.

The notebook:
1. constructs the broadened photon and matter continua,
2. evaluates the analytic principal-value response functions,
3. builds the exact coupled-continuum denominator,
4. computes gauge-invariant photon and matter spectra,
5. compares them with the lossless Hopfield polariton frequencies,
6. sweeps coupling and damping,
7. optionally verifies the principal-value functions numerically from their defining integrals.


## 1. Physics model

The Power–Zienau–Woolley Hamiltonian contains the diamagnetic term. Its matter frequency is therefore renormalized as

$$
\tilde\omega_x^2=\omega_x^2+4g^2.
$$

For Lorentzian photon and matter reservoirs,

$$
|\zeta(\omega)|^2=
\frac{2\gamma_P\omega^3/\pi}
{(\omega^2-\omega_c^2)^2+\gamma_P^2\omega^2},
$$

$$
|\eta(\omega)|^2=
\frac{2\gamma_M\omega/\pi}
{(\omega^2-\tilde\omega_x^2)^2+\gamma_M^2\omega^2}.
$$

The associated principal-value response functions are

$$
W(\omega)=
2\frac{\omega_c^2(\omega^2-\omega_c^2)-\gamma_P^2\omega^2}
{(\omega^2-\omega_c^2)^2+\gamma_P^2\omega^2},
$$

$$
Z(\omega)=
2\frac{\omega^2-\tilde\omega_x^2}
{(\omega^2-\tilde\omega_x^2)^2+\gamma_M^2\omega^2}.
$$

Define

$$
\mathcal D(\omega)=
\left|
1-g^2\left[W+i\pi|\zeta|^2\right]
\left[Z+i\pi|\eta|^2\right]
\right|^2.
$$

The gauge-invariant photon and matter spectral weights used below are

$$
S_P=
\frac{|\zeta|^2+
g^2|\eta|^2\left(W^2+\pi^2|\zeta|^4\right)}
{\mathcal D},
$$

$$
S_M=
\frac{|\eta|^2+
g^2|\zeta|^2\left(Z^2+\pi^2|\eta|^4\right)}
{\mathcal D}.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


## 2. Parameters

All frequencies must use one consistent unit system. You can interpret the unit as normalized frequency, angular THz, or energy divided by $\hbar$; do not mix conventions.


In [ ]:
# Bare frequencies
wc = 1.00          # cavity/photon frequency
wx = 1.00          # bare matter frequency

# Light-matter coupling
g = 0.15

# Reservoir damping
gamma_P = 0.03
gamma_M = 0.05

# Frequency grid
wmin = 0.02
wmax = 1.60
N = 12000
w = np.linspace(wmin, wmax, N)

# PZW/diamagnetic matter renormalization
wx_tilde = np.sqrt(wx**2 + 4*g**2)

print("wc       =", wc)
print("wx       =", wx)
print("g        =", g)
print("gamma_P  =", gamma_P)
print("gamma_M  =", gamma_M)
print("wx_tilde =", wx_tilde)


## 3. Bare continuum functions

In [ ]:
def zeta_abs2(w, wc, gamma_P):
    # |zeta(w)|^2
    den = (w**2 - wc**2)**2 + (gamma_P*w)**2
    return (2*gamma_P*w**3/np.pi) / den


def eta_abs2(w, wx_tilde, gamma_M):
    # |eta(w)|^2
    den = (w**2 - wx_tilde**2)**2 + (gamma_M*w)**2
    return (2*gamma_M*w/np.pi) / den


def W_response(w, wc, gamma_P):
    # Analytic principal-value response W(w)
    den = (w**2 - wc**2)**2 + (gamma_P*w)**2
    return 2*(wc**2*(w**2 - wc**2) - gamma_P**2*w**2) / den


def Z_response(w, wx_tilde, gamma_M):
    # Analytic principal-value response Z(w)
    den = (w**2 - wx_tilde**2)**2 + (gamma_M*w)**2
    return 2*(w**2 - wx_tilde**2) / den


A = zeta_abs2(w, wc, gamma_P)
B = eta_abs2(w, wx_tilde, gamma_M)
W = W_response(w, wc, gamma_P)
Z = Z_response(w, wx_tilde, gamma_M)


In [ ]:
fig, ax = plt.subplots()
ax.plot(w, A, label=r"$|\\zeta(\\omega)|^2$")
ax.plot(w, B, label=r"$|\\eta(\\omega)|^2$")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("spectral weight")
ax.set_title("Bare photon and matter continua")
ax.legend()
plt.show()


## 4. Exact coupled-continuum denominator

Introduce the complex response factors

$$
R_P=W+i\pi|\zeta|^2,
\qquad
R_M=Z+i\pi|\eta|^2.
$$

Then

$$
D_c=1-g^2R_PR_M,
\qquad
\mathcal D=|D_c|^2.
$$

Keeping this as a complex number is numerically safer than manually expanding all real and imaginary terms.


In [ ]:
R_P = W + 1j*np.pi*A
R_M = Z + 1j*np.pi*B

D_complex = 1.0 - g**2 * R_P * R_M
D = np.abs(D_complex)**2
D_safe = np.maximum(D, 1e-30)


## 5. Exact gauge-invariant spectra

The continuum calculation has a gauge freedom because the two reservoir continua are degenerate. Individual branch amplitudes can therefore depend on the chosen continuum basis.

The sums over the two branches are gauge invariant. Numerically we work directly with those gauge-invariant quantities.


In [ ]:
S_ph = (A + g**2 * B * (W**2 + (np.pi*A)**2)) / D_safe
S_mat = (B + g**2 * A * (Z**2 + (np.pi*B)**2)) / D_safe

fig, ax = plt.subplots()
ax.plot(w, S_ph, label="photon spectral weight")
ax.plot(w, S_mat, label="matter spectral weight")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("spectral weight")
ax.set_title("Exact coupled lossy polariton spectra")
ax.legend()
plt.show()


## 6. Dispersive versus absorptive response

In [ ]:
fig, ax = plt.subplots()
ax.plot(w, W, label=r"$W(\\omega)$")
ax.plot(w, Z, label=r"$Z(\\omega)$")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("dispersive response")
ax.set_title("Principal-value/dispersive parts")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(w, np.pi*A, label=r"$\\pi|\\zeta|^2$")
ax.plot(w, np.pi*B, label=r"$\\pi|\\eta|^2$")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("absorptive response")
ax.set_title("Absorptive parts")
ax.legend()
plt.show()


## 7. Lossless Hopfield reference

In the lossless case the polariton frequencies obey

$$
(\omega^2-\omega_c^2)(\omega^2-\tilde\omega_x^2)
=4g^2\omega_c^2.
$$

Thus

$$
\omega_{\pm}=
\frac{1}{\sqrt2}
\sqrt{
\omega_c^2+\tilde\omega_x^2
\pm
\sqrt{(\omega_c^2-\tilde\omega_x^2)^2+16g^2\omega_c^2}
}.
$$


In [ ]:
disc = np.sqrt((wc**2 - wx_tilde**2)**2 + 16*g**2*wc**2)
w_minus = np.sqrt(0.5*(wc**2 + wx_tilde**2 - disc))
w_plus  = np.sqrt(0.5*(wc**2 + wx_tilde**2 + disc))

print(f"Lower polariton reference = {w_minus:.6f}")
print(f"Upper polariton reference = {w_plus:.6f}")

fig, ax = plt.subplots()
ax.plot(w, S_ph, label="photon spectrum")
ax.plot(w, S_mat, label="matter spectrum")
ax.axvline(w_minus, linestyle="--", label="lossless LP")
ax.axvline(w_plus, linestyle="--", label="lossless UP")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("spectral weight")
ax.set_title("Lossy spectrum and lossless Hopfield reference")
ax.legend()
plt.show()


## 8. Locate the two dominant spectral peaks

In [ ]:
def local_peak_indices(y):
    # Very simple peak detector: point higher than immediate neighbors.
    return np.where((y[1:-1] > y[:-2]) & (y[1:-1] > y[2:]))[0] + 1


peaks = local_peak_indices(S_ph)
peaks = peaks[np.argsort(S_ph[peaks])[::-1]]

print("Largest photon-spectrum peaks:")
for idx in peaks[:6]:
    print(f"omega = {w[idx]:.6f}, S_ph = {S_ph[idx]:.6g}")


## 9. Coupling sweep

In [ ]:
g_values = [0.03, 0.08, 0.15, 0.25]

fig, ax = plt.subplots()

for gg in g_values:
    wx_t = np.sqrt(wx**2 + 4*gg**2)
    AA = zeta_abs2(w, wc, gamma_P)
    BB = eta_abs2(w, wx_t, gamma_M)
    WW = W_response(w, wc, gamma_P)
    ZZ = Z_response(w, wx_t, gamma_M)

    Dc = 1 - gg**2*(WW + 1j*np.pi*AA)*(ZZ + 1j*np.pi*BB)
    DD = np.abs(Dc)**2

    S = (AA + gg**2*BB*(WW**2 + (np.pi*AA)**2)) / np.maximum(DD, 1e-30)
    ax.plot(w, S, label=fr"$g={gg}$")

ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("photon spectral weight")
ax.set_title("Evolution with light-matter coupling")
ax.legend()
plt.show()


## 10. Photon-loss / matter-loss sweep

In [ ]:
loss_pairs = [
    (0.01, 0.01),
    (0.03, 0.03),
    (0.08, 0.03),
    (0.03, 0.10),
]

fig, ax = plt.subplots()

for gp, gm in loss_pairs:
    wx_t = np.sqrt(wx**2 + 4*g**2)
    AA = zeta_abs2(w, wc, gp)
    BB = eta_abs2(w, wx_t, gm)
    WW = W_response(w, wc, gp)
    ZZ = Z_response(w, wx_t, gm)

    Dc = 1 - g**2*(WW + 1j*np.pi*AA)*(ZZ + 1j*np.pi*BB)
    DD = np.abs(Dc)**2

    S = (AA + g**2*BB*(WW**2 + (np.pi*AA)**2)) / np.maximum(DD, 1e-30)
    ax.plot(w, S, label=fr"$\\gamma_P={gp},\\ \\gamma_M={gm}$")

ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("photon spectral weight")
ax.set_title("Effect of frequency-independent Lorentzian damping")
ax.legend()
plt.show()


## 11. Optional numerical principal-value check

The analytic $W$ and $Z$ above are the efficient formulas. To see where they come from, one can evaluate the defining principal-value integrals numerically.

For example,

$$
W(\omega)=
P\int_0^\infty
d\omega'
\frac{2\omega'}{\omega^2-\omega'^2}
|\zeta(\omega')|^2,
$$

and

$$
Z(\omega)=
P\int_0^\infty
d\omega'
\frac{2\omega'}{\omega^2-\omega'^2}
|\eta(\omega')|^2.
$$

The integral is singular at $\omega'=\omega$. We therefore split the integral around that point. This is deliberately a simple implementation, not an optimized Hilbert-transform routine.


In [ ]:
def pv_integral_response(omega0, spectral_function, xmax=8.0, nside=6000,
                         exclusion=2e-4):
    # Principal-value integral:
    # integral_0^infinity 2*w'/(omega0^2-w'^2) * spectral_function(w') dw'
    #
    # We truncate infinity at xmax and remove a symmetric small interval
    # around the pole. Increase xmax and nside to test convergence.

    left_end = max(1e-8, omega0 - exclusion)
    right_start = omega0 + exclusion

    left = np.linspace(1e-8, left_end, nside)
    right = np.linspace(right_start, xmax, nside)

    f_left = (2*left/(omega0**2 - left**2)) * spectral_function(left)
    f_right = (2*right/(omega0**2 - right**2)) * spectral_function(right)

    return np.trapezoid(f_left, left) + np.trapezoid(f_right, right)


# Check only a few frequencies because the direct PV integral is intentionally expensive.
test_omegas = [0.70, 0.90, 1.10, 1.30]

print("omega    W_analytic    W_PV_numeric    Z_analytic    Z_PV_numeric")
for ww in test_omegas:
    Wa = W_response(np.array([ww]), wc, gamma_P)[0]
    Za = Z_response(np.array([ww]), wx_tilde, gamma_M)[0]

    Wn = pv_integral_response(
        ww, lambda x: zeta_abs2(x, wc, gamma_P),
        xmax=8.0, nside=4000, exclusion=2e-4
    )
    Zn = pv_integral_response(
        ww, lambda x: eta_abs2(x, wx_tilde, gamma_M),
        xmax=8.0, nside=4000, exclusion=2e-4
    )

    print(f"{ww:5.2f}   {Wa:12.6f}   {Wn:13.6f}   {Za:12.6f}   {Zn:13.6f}")


### Important numerical note

The direct PV integral above is mainly pedagogical. Convergence depends on the cutoff, grid density, and excluded interval around the pole.

For production calculations, use the analytic Lorentzian expressions when the bath is Lorentzian. For an arbitrary frequency-dependent reservoir, the same continuum equations can be evaluated numerically with a proper principal-value/Hilbert-transform method.


## 12. Simple non-Hermitian comparison

A common approximate model replaces the full reservoirs by complex frequencies. One illustrative convention is

$$
H_{\rm eff}=
\begin{pmatrix}
\omega_c-i\gamma_P/2 & g\\
g & \omega_x-i\gamma_M/2
\end{pmatrix}.
$$

Its complex eigenvalues are easy to compute, but they are **not** the exact frequency-dependent-reservoir solution above. This cell is included only as a comparison.


In [ ]:
Heff = np.array([
    [wc - 1j*gamma_P/2, g],
    [g, wx - 1j*gamma_M/2]
], dtype=complex)

eigvals = np.linalg.eigvals(Heff)
eigvals = eigvals[np.argsort(eigvals.real)]

print("Approximate non-Hermitian eigenfrequencies:")
for ev in eigvals:
    print(f"{ev.real:.6f} {ev.imag:+.6f} i")


## 13. One-function reusable simulator

The next cell packages the main calculation so you can change parameters without rewriting the notebook.


In [ ]:
def simulate_lossy_polariton(
    wc=1.0, wx=1.0, g=0.15,
    gamma_P=0.03, gamma_M=0.05,
    wmin=0.02, wmax=1.60, N=12000
):
    w = np.linspace(wmin, wmax, N)
    wx_tilde = np.sqrt(wx**2 + 4*g**2)

    A = zeta_abs2(w, wc, gamma_P)
    B = eta_abs2(w, wx_tilde, gamma_M)
    W = W_response(w, wc, gamma_P)
    Z = Z_response(w, wx_tilde, gamma_M)

    RP = W + 1j*np.pi*A
    RM = Z + 1j*np.pi*B

    Dc = 1 - g**2*RP*RM
    D = np.abs(Dc)**2

    S_ph = (A + g**2*B*(W**2 + (np.pi*A)**2)) / np.maximum(D, 1e-30)
    S_mat = (B + g**2*A*(Z**2 + (np.pi*B)**2)) / np.maximum(D, 1e-30)

    disc = np.sqrt((wc**2 - wx_tilde**2)**2 + 16*g**2*wc**2)
    wp = np.sqrt(0.5*(wc**2 + wx_tilde**2 + disc))
    wm = np.sqrt(0.5*(wc**2 + wx_tilde**2 - disc))

    return {
        "w": w,
        "zeta_abs2": A,
        "eta_abs2": B,
        "W": W,
        "Z": Z,
        "D_complex": Dc,
        "D": D,
        "S_photon": S_ph,
        "S_matter": S_mat,
        "wx_tilde": wx_tilde,
        "omega_LP_lossless": wm,
        "omega_UP_lossless": wp,
    }


# Example:
result = simulate_lossy_polariton(
    wc=1.0, wx=1.0, g=0.18,
    gamma_P=0.025, gamma_M=0.05
)

print("LP =", result["omega_LP_lossless"])
print("UP =", result["omega_UP_lossless"])

fig, ax = plt.subplots()
ax.plot(result["w"], result["S_photon"], label="photon")
ax.plot(result["w"], result["S_matter"], label="matter")
ax.set_xlabel(r"$\\omega$")
ax.set_ylabel("spectral weight")
ax.legend()
plt.show()


## 14. What this simulation demonstrates

- The photon and matter degrees of freedom are coupled through their **full frequency-dependent continua**.
- Loss produces both absorptive and dispersive contributions.
- The diamagnetic term changes the matter frequency from $\omega_x$ to $\tilde\omega_x$.
- The exact spectrum is obtained from the coupled-continuum denominator rather than simply inserting a constant imaginary part into the bare frequencies.
- In the lossless limit the resonances approach the Hopfield polariton frequencies.
- For arbitrary reservoirs, the analytic Lorentzian formulas for $W$ and $Z$ can be replaced by numerical principal-value integrals while keeping the rest of the implementation essentially unchanged.

### Suggested experiments

1. Set `gamma_P = gamma_M = 0.005` and compare with the lossless frequencies.
2. Increase `g` from `0.02` to `0.30` and observe the splitting.
3. Keep `gamma_P` small and increase `gamma_M` to see matter-induced broadening.
4. Detune the system by changing `wx` from `0.7` to `1.3`.
5. Replace the Lorentzian reservoir functions with your own numerically sampled $|\zeta(\omega)|^2$ and $|\eta(\omega)|^2$, then evaluate the PV functions numerically.
